# Infinity-2B Step-wise PFB vs Content-Orthogonal (Colab)

This notebook mounts a full `VAR-SOICT` folder from Google Drive, caches Infinity-2B Q8 GGUF, Infinity VAE, and FLAN-T5-XL encoder under `working_dir/model_dir`, then saves step-wise comparisons: **Baseline | PFB | Content-Orthogonal**.

Use a GPU runtime. An A100/L4 with sufficient VRAM is recommended.

## 1. Mount Drive and choose the working directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys

# Change only this line if the uploaded folder has another name/location.
WORKING_DIR = Path('/content/drive/MyDrive/VAR-SOICT').resolve()
MODEL_DIR = WORKING_DIR / 'model_dir'
OUTPUT_DIR = WORKING_DIR / 'outputs' / 'stepwise_pfb_content_ortho'

if not (WORKING_DIR / 'src' / 'var_soict').exists():
    raise FileNotFoundError(f'VAR-SOICT source not found under {WORKING_DIR}')
if not (WORKING_DIR / 'Infinity' / 'infinity' / 'models' / 'infinity.py').exists():
    raise FileNotFoundError(f'Vendored Infinity source not found under {WORKING_DIR / "Infinity"}')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORKING_DIR / 'src'))

print('Working directory:', WORKING_DIR)
print('Persistent model cache:', MODEL_DIR)
print('Persistent outputs:', OUTPUT_DIR)

## 2. Check GPU and install runtime dependencies

Do **not** run `pip install flash_attn` or install an old unmodified `Infinity/requirements.txt`. Current Colab Python 3.13 generally has no compatible FlashAttention wheel; this repository uses PyTorch SDPA instead.

In [ ]:
from var_soict.bootstrap import check_torch_runtime, install_dependencies

DEVICE = check_torch_runtime(require_cuda=True)
install_dependencies()

# Added explicitly because bootstrap uses hf_hub_download for the three checkpoints.
%pip install -q huggingface_hub

## 3. Configure Drive-backed model cache

In [ ]:
from var_soict.config import ExperimentConfig

config = ExperimentConfig(
    root=MODEL_DIR,
    infinity_source_dir=WORKING_DIR / 'Infinity',
    download_missing_model_files=True,
    output_run_name='stepwise_pfb_content_ortho',
    model_pn='0.25M',
    cfg=1.0,
    tau=0.1,
    top_k=600,
    top_p=0.95,
    seed=2026,
    t5_device='cuda',
)
print('Infinity source:', WORKING_DIR / 'Infinity')
print('Weights/cache only:', MODEL_DIR)

## 4. Download/cache Infinity-2B, Infinity VAE, and FLAN-T5-XL

In [ ]:
from var_soict.bootstrap import (
    import_gguf_loader_direct,
    prepare_direct_model_files,
)

INFINITY_SOURCE_DIR = WORKING_DIR / 'Infinity'
model_files = prepare_direct_model_files(
    config,
    model_dir=MODEL_DIR,
    infinity_source_dir=INFINITY_SOURCE_DIR,
)
gguf_loader = import_gguf_loader_direct(
    model_files,
    infinity_source_dir=INFINITY_SOURCE_DIR,
)

print('Infinity-2B:', model_files.infinity_gguf)
print('Infinity VAE:', model_files.vae_path)
print('FLAN-T5-XL:', model_files.t5_gguf)
print('No Infinity_runtime directory is used.')

## 5. Initialize the Infinity model

In [ ]:
from var_soict.bootstrap import build_scale_schedule, load_model_bundle

bundle = load_model_bundle(config, model_files, gguf_loader)
bundle.scale_schedule = build_scale_schedule(config.model_pn, aspect_ratio=1.0)

assert hasattr(bundle.infinity_model, 'autoregressive_infer_pfb')
assert hasattr(bundle.infinity_model, 'autoregressive_infer_content_ortho')
print('Number of AR resolutions:', len(bundle.scale_schedule))

## 6. Select a CSD100 content/style pair

The styled prompt deliberately keeps the content subject unchanged and adds only the style phrase. The CSD100 style image is retained as a visual reference.

In [ ]:
from IPython.display import display
from PIL import Image

from var_soict.csd100_stepwise import load_csd100_stepwise_case

# Folder names under WORKING_DIR/csd100. Change these two values freely.
CONTENT_ITEM_ID = 'fox+graffiti'
STYLE_ITEM_ID = 'pen+artwork'

case = load_csd100_stepwise_case(
    WORKING_DIR,
    content_item_id=CONTENT_ITEM_ID,
    style_item_id=STYLE_ITEM_ID,
)
print('Content prompt:', case.content_prompt)
print('Styled prompt :', case.style_prompt)
print('Style label   :', case.style_label)
display(Image.open(case.content_reference_path).convert('RGB'))
display(Image.open(case.style_reference_path).convert('RGB'))

## 7. Run the step-wise experiment

Start with a few diagnostically useful steps. Set `INJECT_STEPS = None` to run every resolution. Each selected step performs one PFB run and one content-orthogonal run.

In [ ]:
from var_soict.csd100_stepwise import run_csd100_stepwise

INJECT_STEPS = [0, 1, 2, 3, 6, 9]
SAC = False  # Rerun with True after inspecting the residual-only experiment.

case, result = run_csd100_stepwise(
    bundle,
    config,
    var_soict_root=WORKING_DIR,
    content_item_id=CONTENT_ITEM_ID,
    style_item_id=STYLE_ITEM_ID,
    output_root=OUTPUT_DIR,
    inject_steps=INJECT_STEPS,
    seed=config.seed,
    sac=SAC,
    style_rank=1,
    content_rank=1,
    strength=1.0,
    projection_strength=1.0,
)
print('Case output:', OUTPUT_DIR / case.case_name)

## 8. Display Baseline | PFB | Content-Orthogonal comparisons

In [ ]:
from var_soict.stepwise_feature_experiment import display_stepwise_comparisons

display_stepwise_comparisons(result)

## 9. Inspect saved artifacts

In [ ]:
case_output_dir = OUTPUT_DIR / case.case_name
for path in sorted(case_output_dir.iterdir()):
    print(path.name)